# Appending grades from .JSONL to df

In [1]:
# ==== IMPORTS ====
import pandas as pd
import json


# ==== SETTINGS ====
# ---- IMPORT SETTINGS ----
IMPORT_PATH = "../../data/gcp_manual_copy/"

# ---- EXPORT SETTINGS ----
EXPORT = True
EXPORT_PATH = IMPORT_PATH

# ---- FILES ----
# Metrics files
FILE_ALL = "thesis_meta_all_metrics_except_grade_27032026.parquet"
FILE_FILTERED = "thesis_meta_all_metrics_except_grade_filtered_27032026.parquet"

# Grade file
#JSONL_NAME = "dtu_findit-extraction_and_processing-grading_results.jsonl"
JSONL_NAME = "dtu_findit-extraction_and_processing-grading_results_more_but_sketchy.jsonl"

## Loading df dataset

In [2]:
# ==== FUNCTION ====
def load_csv_to_df(csv_path, sep=";"):
    try:
        df = pd.read_csv(csv_path, encoding="utf-8", sep=sep)
        print(f"Successfully loaded CSV from {csv_path}")
        print(f"DataFrame shape: {df.shape}")
        print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading CSV from {csv_path}: {e}")
        return None

def load_parquet_to_df(parquet_path, na=False):
    try:
        df = pd.read_parquet(parquet_path)
        print(f"Successfully loaded Parquet from {parquet_path}")
        print(f"DataFrame shape: {df.shape}")
        if na:
            print(f"DataFrame N/A counts:\n{df.isna().sum()}\n")
        print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading Parquet from {parquet_path}: {e}")
        return None

# ==== LOAD DATAFRAMES ====
df_all = load_parquet_to_df(IMPORT_PATH + FILE_ALL)
df_filtered = load_parquet_to_df(IMPORT_PATH + FILE_FILTERED)

# ==== DROP NA COLUMNS ====
df_all_noNA = df_all.dropna(axis=1, how="all")
print(f"df_all_noNA shape: {df_all_noNA.shape}")
print(f"df_all_noNA columns: {df_all_noNA.columns.tolist()}\n")
df_filtered_noNA = df_filtered.dropna(axis=1, how="all")
print(f"df_filtered_noNA shape: {df_filtered_noNA.shape}")
print(f"df_filtered_noNA columns: {df_filtered_noNA.columns.tolist()}\n")

# ==== COLUMNS TO DROP ====
drop_columns = [
    "access_ss",
    "Affiliations",
    "collection_facet",
    "format",
    "fulltext_availability_facet",
    "ISBN",
    "Journal Page",
    "isolanguage_facet",
    "Publisher",
    "Source",
    "source_all_ss",
    "match_trigger",
    "equation_pipeline_version",
    "pdf_file_analysis",
    "num_tot_pages_analysis",
    "num_cont_pages_analysis",
    "num_words_full_analysis",
    "num_words_cont_analysis",
    "abstract_ts_analysis",
    "Author_analysis",
    "Publication Year_analysis",
    "primary_member_id_s_analysis",
    "Title_analysis",
    ]

# ==== DROP COLUMNS ====
df_all_noNA_clean = df_all_noNA.drop(columns=drop_columns, errors="ignore")
print(f"df_all_noNA_clean shape: {df_all_noNA_clean.shape}")
print(f"df_all_noNA_clean columns: {df_all_noNA_clean.columns.tolist()}\n")
df_filtered_noNA_clean = df_filtered_noNA.drop(columns=drop_columns, errors="ignore")
print(f"df_filtered_noNA_clean shape: {df_filtered_noNA_clean.shape}")
print(f"df_filtered_noNA_clean columns: {df_filtered_noNA_clean.columns.tolist()}\n")

# ==== LOCK DATAFRAMES FOR ANALYSIS ====
print("Final DataFrames for Analysis:")
df_all_final = df_all_noNA_clean.copy()
print(f"df_all_final shape: {df_all_final.shape}")
df_filtered_final = df_filtered_noNA_clean.copy()
print(f"df_filtered_final shape: {df_filtered_final.shape}")

Successfully loaded Parquet from ../../data/gcp_manual_copy/thesis_meta_all_metrics_except_grade_27032026.parquet
DataFrame shape: (19690, 76)
DataFrame columns: ['abstract_ts', 'access_ss', 'Affiliations', 'Timestamp', 'Author', 'citation_count_i', 'ID', 'dtu_library_collection_facet', 'collection_facet', 'Publication Year', 'Conference', 'DOI', 'Editor', 'embargo_ssf', 'format', 'fulltext_availability_facet', 'has_openaccess_fulltext_b', 'holdings_ssf', 'ISBN', 'Journal Issue', 'journal_issue_tsort', 'journal_oa_model_ss', 'Journal Page', 'journal_page_start_tsort', 'Journal Title', 'journal_title_facet', 'toc_key_s', 'Journal Volume', 'journal_vol_tsort', 'keywords_ts', 'keywords_facet', 'keywords_normalized', 'isolanguage_facet', 'member_id_ss', 'ORCID', 'primary_member_id_s', 'Publisher', 'Source', 'source_all_ss', 'Title', 'MASTER THESIS TITLE', 'BY', 'SUPERVISED BY', 'YEAR', 'PUBLISHER', 'TYPES', 'pdf_file', 'num_tot_pages', 'num_cont_pages', 'num_words_full', 'num_words_cont', 

## Loading .JSONL

In [3]:
# ==== LOAD .JSONL ====
# Load the lines manually to handle the nested structure effectively
data = []
with open(IMPORT_PATH + JSONL_NAME, 'r') as f:
    for line in f:
        data.append(json.loads(line))

# Flatten the 'data' and 'meta' keys into the main dataframe
df_grades = pd.json_normalize(data)

# Resulting columns will look like: 'data.scientific_contribution', 'meta.attempts', etc.
print(df_grades.shape)
display(df_grades.head())

(6281, 23)


,file,analysis,data.scientific_contribution,data.methodological_rigor,data.technical_implementation,data.literature_review,data.process_professionalism,data.impact_applicability,data.research_question_alignment,data.total_score,...,meta.input_chars,meta.estimated_input_tokens,meta.max_input_chars_effective,meta.was_truncated,meta.prompt_fit_attempts,meta.timeout_seconds,meta.stream_mode,data,meta.trim_marker,data.error
0,5e4bd47fd9001d2ceb5bbad0_Mapping the potential...,"{\n ""scientific_contribution"": 14,\n ""method...",14.0,10.0,12.0,2.0,5.0,5.0,6.0,54.0,...,13258,4017,NaN,False,1,6000,openai_compat_stream,NaN,NaN,NaN
1,5f46438bd9001d01721d7e29_Effect of probiotic T...,"{\n ""scientific_contribution"": 14,\n ""method...",14.0,14.0,15.0,8.0,7.0,8.0,8.0,74.0,...,20040,6072,NaN,False,1,6000,openai_compat_stream,NaN,NaN,NaN
2,60364261d9001d01656e1c2e_Visual Navigation for...,"{\n ""scientific_contribution"": 14,\n ""method...",14.0,15.0,14.0,8.0,8.0,7.0,8.0,74.0,...,23939,7254,NaN,False,1,6000,openai_compat_stream,NaN,NaN,NaN
3,5e3419a8d9001d01573c151b_Classification of vir...,"{\n ""scientific_contribution"": 15,\n ""method...",15.0,13.0,14.0,8.0,9.0,7.0,8.0,74.0,...,30140,9133,NaN,False,1,6000,openai_compat_stream,NaN,NaN,NaN
4,5f576705d9001d01706cbbcb_Investigation of aero...,"{\n ""scientific_contribution"": 12,\n ""method...",12.0,13.0,14.0,8.0,8.0,6.0,9.0,70.0,...,34004,10304,NaN,False,1,6000,openai_compat_stream,NaN,NaN,NaN


In [4]:
# ==== PREPARE GRADE COLUMNS FOR LEFT JOIN ====
grade_column_map = {
    "data.scientific_contribution": "grading_scientific_contribution",
    "data.methodological_rigor": "grading_methodological_rigor",
    "data.technical_implementation": "grading_technical_implementation",
    "data.literature_review": "grading_literature_review",
    "data.process_professionalism": "grading_process_professionalism",
    "data.impact_applicability": "grading_impact_applicability",
    "data.research_question_alignment": "grading_research_question_alignment",
    "data.total_score": "grading_total_score",
    "meta.attempts": "grading_meta_attempts",
    "meta.original_chars": "grading_meta_original_chars",
    "meta.trimmed_at_references": "grading_meta_trimmed_at_references",
    "meta.input_chars": "grading_meta_input_chars",
    "meta.estimated_input_tokens": "grading_meta_estimated_input_tokens",
    "meta.was_truncated": "grading_meta_was_truncated",
    "meta.prompt_fit_attempts": "grading_meta_prompt_fit_attempts",
    "meta.timeout_seconds": "grading_meta_timeout_seconds",
    "meta.stream_mode": "grading_meta_stream_mode",
}

required_grade_cols = ["file"] + list(grade_column_map.keys())
missing_in_grades = [c for c in required_grade_cols if c not in df_grades.columns]
if missing_in_grades:
    raise KeyError(f"Missing required columns in df_grades: {missing_in_grades}")

# Keep only relevant columns and rename to target grading_* names
df_grades_for_join = df_grades[required_grade_cols].rename(columns=grade_column_map).copy()

# Normalize join keys to reduce mismatch risk from whitespace
df_grades_for_join["file"] = df_grades_for_join["file"].astype(str).str.strip()
df_filtered_final["pdf_file"] = df_filtered_final["pdf_file"].astype(str).str.strip()

# Detect duplicate grade keys and keep the row with the most non-null grade/meta values
target_grade_cols = list(grade_column_map.values())
dup_count = df_grades_for_join["file"].duplicated().sum()
if dup_count > 0:
    df_grades_for_join["_grade_non_null_count"] = df_grades_for_join[target_grade_cols].notna().sum(axis=1)
    df_grades_for_join = (
        df_grades_for_join
        .sort_values(["file", "_grade_non_null_count"], ascending=[True, False])
        .drop_duplicates(subset=["file"], keep="first")
        .drop(columns=["_grade_non_null_count"])
    )
    print(
        f"Duplicate file keys found in df_grades: {dup_count}. "
        "Kept the most complete row per file key (highest non-null grade/meta count)."
    )

# If target grading columns already exist on the left, drop them before merge to avoid _x/_y suffixes
existing_target_cols = [c for c in target_grade_cols if c in df_filtered_final.columns]
if existing_target_cols:
    print(f"Dropping existing grading columns before merge: {existing_target_cols}")
    df_filtered_base = df_filtered_final.drop(columns=existing_target_cols).copy()
else:
    df_filtered_base = df_filtered_final.copy()

# ==== LEFT JOIN ENRICHMENT ====
df_filtered_final_enriched = df_filtered_base.merge(
    df_grades_for_join,
    how="left",
    left_on="pdf_file",
    right_on="file",
    validate="m:1",
).drop(columns=["file"] )

print(f"df_filtered_final_enriched shape: {df_filtered_final_enriched.shape}. Compared to df_filtered_final shape: {df_filtered_final.shape}.")
print("Added grading columns:")
print(target_grade_cols)

# Quick match diagnostics
score_col = "grading_total_score" if "grading_total_score" in df_filtered_final_enriched.columns else None
if score_col is None:
    available = [c for c in df_filtered_final_enriched.columns if "total_score" in c]
    raise KeyError(
        "Could not find grading_total_score after merge. "
        f"Available total_score-like columns: {available}"
    )

matched_rows = df_filtered_final_enriched[score_col].notna().sum()
total_rows = len(df_filtered_final_enriched)
print(f"Matched rows with grades: {matched_rows}/{total_rows} ({matched_rows/total_rows:.1%})")

display(df_filtered_final_enriched.head())

Duplicate file keys found in df_grades: 106. Kept the most complete row per file key (highest non-null grade/meta count).
Dropping existing grading columns before merge: ['grading_scientific_contribution', 'grading_methodological_rigor', 'grading_technical_implementation', 'grading_literature_review', 'grading_process_professionalism', 'grading_impact_applicability', 'grading_research_question_alignment', 'grading_total_score', 'grading_meta_attempts', 'grading_meta_original_chars', 'grading_meta_trimmed_at_references', 'grading_meta_input_chars', 'grading_meta_estimated_input_tokens', 'grading_meta_was_truncated', 'grading_meta_prompt_fit_attempts', 'grading_meta_timeout_seconds', 'grading_meta_stream_mode']
df_filtered_final_enriched shape: (6254, 45). Compared to df_filtered_final shape: (6254, 45).
Added grading columns:
['grading_scientific_contribution', 'grading_methodological_rigor', 'grading_technical_implementation', 'grading_literature_review', 'grading_process_professionali

,Timestamp,Author,ID,Publication Year,member_id_ss,primary_member_id_s,Title,MASTER THESIS TITLE,BY,SUPERVISED BY,...,grading_total_score,grading_meta_attempts,grading_meta_original_chars,grading_meta_trimmed_at_references,grading_meta_input_chars,grading_meta_estimated_input_tokens,grading_meta_was_truncated,grading_meta_prompt_fit_attempts,grading_meta_timeout_seconds,grading_meta_stream_mode
0,2025-02-26T01:51:41.171Z,"Uth, Matilde",2868333190,2025,67be6a2d56d32566d74defce,67be6a2d56d32566d74defce,Comparing long and short read technologies in ...,Comparing long and short read technologies in ...,"Uth, Matilde","Otani, Saria; Petersen, Thomas Nordahl",...,87.0,1.0,83419.0,False,83417.0,25277.0,False,1.0,600.0,openai_compat_stream
1,2025-02-26T01:51:40.290Z,"Gravesen, Johannes Bøgelund",2868333131,2025,67be6a16754a9d66d7be5fb3,67be6a16754a9d66d7be5fb3,Effect of Nozzle Geometry on a Water Mist Fire...,Effect of Nozzle Geometry on a Water Mist Fire...,"Gravesen, Johannes Bøgelund","Walther, Jens Honore; Meyer, Knud Erik",...,81.0,1.0,297604.0,False,297542.0,90164.0,False,1.0,600.0,openai_compat_stream
2,2025-08-14T00:09:36.887Z,"Deng, Siyuan",2872988369,2025,689d294027df7f01020cebe5,689d294027df7f01020cebe5,Vision-Based Process Mining: Event Extraction ...,Vision-Based Process Mining: Event Extraction ...,"Deng, Siyuan","López-Acosta, Hugo-Andrés",...,87.0,1.0,156257.0,False,156255.0,47350.0,False,1.0,600.0,openai_compat_stream
3,2025-08-02T00:10:27.278Z,"Gereoudakis, Polyvios",2872712978,2025,688d57728e1e270102bab9b6,688d57728e1e270102bab9b6,Ultrafine Particle Pollution (UFP) in Indoor A...,Ultrafine Particle Pollution (UFP) in Indoor A...,"Gereoudakis, Polyvios","Mikkelsen, Teis Nørgaard",...,79.0,1.0,102638.0,False,102597.0,31090.0,False,1.0,600.0,openai_compat_stream
4,2024-12-18T01:26:35.134Z,"Diez, Enrique Martínez",2854137646,2024,676220d877b049a804be5ce2,676220d877b049a804be5ce2,The role of germline variants in T cells respo...,The role of germline variants in T cells respo...,"Diez, Enrique Martínez","Nos, Grigorii; Hadrup, Sine Reker",...,73.0,1.0,23529.0,False,23528.0,7129.0,False,1.0,600.0,openai_compat_stream


## EXPORTING THE DF

In [5]:
pct_grades = round(matched_rows/total_rows * 100)
date_stamp = pd.Timestamp.now().strftime("%d%m%Y")

# ==== EXPORT ENRICHED DATAFRAME ====
if EXPORT:
    export_name = f"thesis_meta_all_metrics_with_{pct_grades}pct_grades_{date_stamp}.parquet"
    df_filtered_final_enriched.to_parquet(EXPORT_PATH + export_name, index=False)
    print(f"Exported enriched DataFrame to {EXPORT_PATH + export_name}")
else:
    print("Enriched DataFrame not exported.")

Exported enriched DataFrame to ../../data/gcp_manual_copy/thesis_meta_all_metrics_with_96pct_grades_20042026.parquet
